In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# OpenMDAO `Problem` Functional Interface

The functional interface lets you wrap an assembled OpenMDAO `Problem` in a
Python callable so that external tools — optimizers, surrogate trainers,
uncertainty-quantification frameworks, etc. — can drive it using ordinary
NumPy arrays instead of the OpenMDAO variable-name API.

The entry point is `Problem.get_callback()`, which returns a
`_FunctionalCallback` object.  Calling that object

1. takes a flat input vector as an argument and writes it into the problem,
2. calls `Problem.run_model()`, and
3. returns outputs and/or total derivatives as NumPy arrays, depending on
   which *form* was requested.

## The three forms

| `form` | Signature | Returns |
|--------|-----------|----------|
| `'f'` | `cb(x[, y])` | flat output vector `y` |
| `'dfdx'` | `cb(x[, J])` | Jacobian `J` of shape `(n_outputs, n_inputs)` |
| `'fdfdx'` | `cb(x[, y[, J]])` | tuple `(y, J)` |

All three forms accept pre-allocated output arrays as optional arguments to
avoid repeated memory allocation.

## Specifying variables

The `input_vars` and `output_vars` arguments to `get_callback()` control which
model variables are included as inputs and outputs of the callback function.

Each entry in either list may be:

* a **plain string** — the variable name; or
* a **dict** mapping an alias to a metadata dict with optional keys:
  * `'name'` — actual variable name when the alias differs;
  * `'indices'` — a subset of elements, in any format accepted by
    `Problem.get_val()`;
  * `'units'` — the units in which the variable should be expressed.  For
    `form='f'` this controls the units used when reading back output values.
    For the derivative forms (`'dfdx'` and `'fdfdx'`) this scales the
    corresponding rows (output variable) or columns (input variable) of the
    Jacobian so that derivatives are expressed in the requested units.

For the derivative forms (`'dfdx'` and `'fdfdx'`), omitting `input_vars`
causes the callback to use the driver's registered design variables, and
omitting `output_vars` causes it to use the driver's registered responses
(objectives + constraints).

For `form='f'`, both arguments are required.

Variables are packed into the flat vector in the order they appear in the
list, with multi-element variables occupying contiguous slices.

## Helper methods on the callback object

| Method | Description |
|--------|-------------|
| `create_input_vector()` | Allocate a flat input array pre-filled with the current problem values. |
| `create_output_vector()` | Allocate a flat output array pre-filled with the current problem values. |
| `create_jacobian_matrix()` | Allocate a zero Jacobian of shape `(n_outputs, n_inputs)`. |
| `get_input_val(name)` | Return the current flat value of one registered input variable. |
| `get_output_val(name)` | Return the current flat value of one registered output variable. |
| `input_var_names` | List of registered input variable names (or aliases). |
| `output_var_names` | List of registered output variable names (or aliases). |
| `form` | The form string this callback was created with. |

## Example 1: evaluating outputs (`form='f'`)

This example uses everyone's favorite `Paraboloid` component,
$f(x, y) = (x-3)^2 + xy + (y+4)^2 - 3$, to show how to evaluate model
outputs through the functional interface. In this example the OpenMDAO `Model` will consist of just this one `Component`, but the `Model` could be arbitrarily complex.

Both `input_vars` and `output_vars` must be supplied when using `form='f'`.
The callback accepts a flat vector whose elements correspond to the variables
listed in `input_vars` in order, and returns a flat vector whose elements
correspond to `output_vars` in order.

In [ ]:
import numpy as np
import openmdao.api as om
from openmdao.test_suite.components.paraboloid import Paraboloid

prob = om.Problem()
prob.model.add_subsystem('comp', Paraboloid(),
                         promotes_inputs=['x', 'y'],
                         promotes_outputs=['f_xy'])
prob.setup()
prob.final_setup()

# Create a callback that maps [x, y] -> [f_xy].
f = prob.get_callback('f', input_vars=['x', 'y'], output_vars=['f_xy'])

print('input variable names :', f.input_var_names)
print('output variable names:', f.output_var_names)

# create_input_vector() returns a 1-D array pre-filled with the current
# problem values.  Modify it in-place before each call.
x = f.create_input_vector()
x[0] = 3.0   # x
x[1] = -4.0  # y

y = f(x)
print('f(3, -4) =', y[0])

x[0] = 6.6667
x[1] = -7.3333
print('f(6.6667, -7.3333) =', f(x)[0])

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

x[0], x[1] = 8.0, 9.0
assert_near_equal(f(x)[0], (x[0]-3.0)**2 + x[0]*x[1] + (x[1]+4.0)**2 - 3.0)

### Passing a pre-allocated output array

Pass a pre-allocated array as `y` to avoid a memory allocation on every call.
Use `create_output_vector()` to get a correctly-sized array.

In [ ]:
y_buf = f.create_output_vector()  # allocate once

x[0], x[1] = 9.0, 10.0
f(x, y=y_buf)  # result written into y_buf; return value is also y_buf
print('f(9, 10) =', y_buf[0])

### Holding some inputs fixed

List only the variables you want to vary in `input_vars`.  Any variable
omitted from the list keeps whatever value is currently stored in the problem.
Set that fixed value with `Problem.set_val()` before creating the callback.

In [ ]:
prob.set_val('y', -4.0)  # fix y; only x will be varied

f_x_only = prob.get_callback('f', input_vars=['x'], output_vars=['f_xy'])

x1 = f_x_only.create_input_vector()  # length 1: only x
x1[0] = 3.0
print('f(x=3, y=-4) =', f_x_only(x1)[0])

## Example 2: total derivatives (`form='dfdx'`)

Use `form='dfdx'` to obtain the Jacobian $\partial f / \partial x$.  The
callback returns a 2-D array of shape `(n_outputs, n_inputs)`.

When `input_vars` and `output_vars` are both omitted the callback
automatically uses the driver's registered design variables as inputs and its
registered responses (objectives + constraints) as outputs.

In [ ]:
prob2 = om.Problem()
prob2.model.add_subsystem('comp', Paraboloid(),
                          promotes_inputs=['x', 'y'],
                          promotes_outputs=['f_xy'])
prob2.model.add_design_var('x', lower=-50, upper=50)
prob2.model.add_design_var('y', lower=-50, upper=50)
prob2.model.add_objective('f_xy')
prob2.driver = om.ScipyOptimizeDriver()
prob2.setup()
prob2.final_setup()

# No input_vars / output_vars: falls back to driver design vars and responses.
dfdx = prob2.get_callback('dfdx')

print('input  vars:', dfdx.input_var_names)
print('output vars:', dfdx.output_var_names)

x = dfdx.create_input_vector()
x[0] = 1.5  # x
x[1] = 2.5  # y

J = dfdx(x)
print('J =', J)
print('shape:', J.shape)  # (1, 2): one output, two inputs

In [ ]:
J_expected = np.array([[2*(x[0]-3.0) + x[1], x[0] + 2*(x[1]+4.0)]])
assert_near_equal(J, J_expected)

### Passing a pre-allocated Jacobian

Use `create_jacobian_matrix()` to allocate a zero matrix of the right shape,
then pass it as the `J` keyword argument to avoid re-allocation.

In [ ]:
J_buf = dfdx.create_jacobian_matrix()  # shape (1, 2), all zeros

x[0], x[1] = 1.6, 2.6
dfdx(x, J=J_buf)
print('J =', J_buf)

In [ ]:
J_expected = np.array([[2*(x[0]-3.0) + x[1], x[0] + 2*(x[1]+4.0)]])
assert_near_equal(J_buf, J_expected)

## Example 3: outputs and derivatives together (`form='fdfdx'`)

Use `form='fdfdx'` when you need both the function value and the Jacobian from
the same model evaluation.  The callback returns the tuple `(y, J)`.

In [ ]:
fdfdx = prob2.get_callback('fdfdx', input_vars=['x', 'y'],
                            output_vars=['f_xy'])

x = fdfdx.create_input_vector()
x[0] = 3.0
x[1] = -4.0

y, J = fdfdx(x)
print('f(3, -4)  =', y[0])
print('df/dx     =', J[0, 0])
print('df/dy     =', J[0, 1])

In [ ]:
assert_near_equal(y[0], (x[0]-3.0)**2 + x[0]*x[1] + (x[1]+4.0)**2 - 3.0)
assert_near_equal(J[0, 0], 2*(x[0]-3.0) + x[1])
assert_near_equal(J[0, 1], x[0] + 2*(x[1]+4.0))

### Pre-allocating both output arrays

Pass pre-allocated `y` and `J` buffers together to eliminate all allocation.

In [ ]:
y_buf = fdfdx.create_output_vector()
J_buf = fdfdx.create_jacobian_matrix()

x[0], x[1] = 6.6667, -7.3333
fdfdx(x, y=y_buf, J=J_buf)
print('f(6.6667, -7.3333) =', y_buf[0])
print('J =', J_buf)

## Example 4: selecting variable subsets with `indices`

When a model variable is an array, you can select only a subset of its
elements to be included as either an input or output to the functional interface by supplying an `'indices'` key in the variable
metadata dict.  The indices may be an integer list (flat indexing) or a tuple
of arrays (multi-dimensional indexing).

The example below uses a component with two scalar inputs `x` and `y` of
shape `(2,)` and one output `f` of shape `(2,)`.  The callback is configured
to expose only `x[0]` and `f[0]`.

In [ ]:
class QuadComp(om.ExplicitComponent):
    """f[0] = (x[0]+2)^2 + (x[1]+3)^2 - 4,  f[1] = 2*x[0] + 3*x[1]."""

    def setup(self):
        self.add_input('x', val=0.0, shape=(2,))
        self.add_output('f', val=0.0, shape=(2,))

    def setup_partials(self):
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        x = inputs['x']
        outputs['f'][0] = (x[0]+2)**2 + (x[1]+3)**2 - 4
        outputs['f'][1] = 2*x[0] + 3*x[1]


prob3 = om.Problem()
prob3.model.add_subsystem('comp', QuadComp(),
                           promotes_inputs=['x'], promotes_outputs=['f'])
prob3.model.add_design_var('x', lower=-50, upper=50)
prob3.model.add_objective('f', index=0)
prob3.driver = om.ScipyOptimizeDriver()
prob3.setup(force_alloc_complex=True)
prob3.final_setup()
prob3.set_val('x', [-2.0, -3.0])
prob3.run_model()

# Expose only x[0] as input and f[0] as output.
fdfdx3 = prob3.get_callback(
    'fdfdx',
    input_vars=[{'x': {'indices': [0]}}],
    output_vars=[{'f': {'indices': [0]}}],
)

x3 = fdfdx3.create_input_vector()  # length 1
print('x3 =', x3)                  # reflects current prob value of x[0]

f3, J3 = fdfdx3(x3)
print('f[0]         =', f3[0])
print('df[0]/dx[0]  =', J3[0, 0])   # shape (1, 1)

In [ ]:
assert_near_equal(J3[0, 0], 2*(x3[0]+2))

## Example 5: driver variables used automatically

When a `Problem` has design variables and responses registered on the driver,
calling `get_callback('dfdx')` (or `'fdfdx'`) without `input_vars` /
`output_vars` automatically selects those driver variables.  This is
convenient after running an optimization: the callback immediately reflects
the driver's variable set including any index subsets specified on the
constraint or objective.

The `input_var_names` and `output_var_names` properties contain which variables were selected.

In [ ]:
prob4 = om.Problem()
prob4.model.add_subsystem('comp', QuadComp(),
                           promotes_inputs=['x'], promotes_outputs=['f'])
prob4.model.add_design_var('x', lower=-50, upper=50)
prob4.model.add_objective('f', index=0, alias='f_obj')
prob4.model.add_constraint('f', indices=[1], equals=13.0, alias='f_con')
prob4.driver = om.ScipyOptimizeDriver()
prob4.setup(force_alloc_complex=True)
prob4.set_val('x', [0.0, 0.0])
prob4.run_driver()

# Both input_vars and output_vars omitted: uses driver design vars + responses.
fdfdx4 = prob4.get_callback('fdfdx')
print('inputs :', fdfdx4.input_var_names)
print('outputs:', fdfdx4.output_var_names)

x4 = fdfdx4.create_input_vector()
print('x at optimum:', x4)   # pre-filled with the post-optimization values

f4, J4 = fdfdx4(x4)
print('f4 =', f4)
print('J4 =\n', J4)

In [ ]:
# At the optimum x = [2, 3].
assert_near_equal(x4, [2.0, 3.0], tolerance=1e-5)

J4_expected = np.array([
    [2*(x4[0]+2), 2*(x4[1]+3)],  # df_obj/dx
    [2,           3           ],  # df_con/dx
])
assert_near_equal(J4, J4_expected, tolerance=1e-5)

## Inspecting current values after a call

After calling the callback the problem is left in the state corresponding to
the last `x` that was passed in.  The helper methods `get_input_val()` and
`get_output_val()` retrieve the current flat values of any registered
variable without another model evaluation.

In [ ]:
# Continue from Example 3.
x_in = fdfdx.create_input_vector()
x_in[0] = 3.0
x_in[1] = -4.0
fdfdx(x_in)

print('x from callback  :', fdfdx.get_input_val('x'))
print('y from callback  :', fdfdx.get_output_val('f_xy'))

## Example 6: unit conversion in Jacobians

When model variables have physical units the Jacobian entries are expressed in those units.  The functional interface lets you request different
units for any input or output variable by adding a `'units'` key to the
variable metadata dict.  The Jacobian is then automatically scaled so that
derivatives are expressed in the requested units.

Consider a component that computes `f = 2 * x` where `x` is in metres and
`f` is in Newtons.  The native Jacobian is $\partial f / \partial x = 2\ \text{N/m}$.

* Requesting `f` in **kN** scales the output rows by $10^{-3}$, giving
  $0.002\ \text{kN/m}$.
* Requesting `x` in **km** scales the input columns by $10^{3}$, giving
  $2000\ \text{N/km}$.
* Requesting both gives $2\ \text{kN/km}$ (the two scalings cancel).

In [ ]:
class ForceComp(om.ExplicitComponent):
    """f = 2 * x,  x in metres,  f in Newtons."""

    def setup(self):
        self.add_input('x', val=1.0, units='m')
        self.add_output('f', val=0.0, units='N')

    def setup_partials(self):
        self.declare_partials('f', 'x')

    def compute(self, inputs, outputs):
        outputs['f'] = 2.0 * inputs['x']

    def compute_partials(self, inputs, partials):
        partials['f', 'x'] = 2.0


prob5 = om.Problem()
prob5.model.add_subsystem('comp', ForceComp(), promotes=['*'])
prob5.model.add_design_var('x', lower=0.0, upper=10.0)
prob5.model.add_objective('f')
prob5.driver = om.ScipyOptimizeDriver()
prob5.setup()
prob5.final_setup()
prob5.set_val('x', 3.0, units='m')
prob5.run_model()

# --- native units: df[N] / dx[m] ---
dfdx_native = prob5.get_callback('dfdx', input_vars=['x'], output_vars=['f'])
x5 = dfdx_native.create_input_vector()
J_native = dfdx_native(x5)
print(f'df[N]/dx[m]   = {J_native[0, 0]:.4f}  (expected 2.0)')

# --- output in kN: df[kN] / dx[m] ---
dfdx_kN = prob5.get_callback('dfdx',
                              input_vars=['x'],
                              output_vars=[{'f': {'units': 'kN'}}])
J_kN = dfdx_kN(x5)
print(f'df[kN]/dx[m]  = {J_kN[0, 0]:.6f}  (expected 0.002)')

# --- input in km: df[N] / dx[km] ---
dfdx_km = prob5.get_callback('dfdx',
                              input_vars=[{'x': {'units': 'km'}}],
                              output_vars=['f'])
J_km = dfdx_km(x5)
print(f'df[N]/dx[km]  = {J_km[0, 0]:.1f}  (expected 2000.0)')

# --- both: df[kN] / dx[km] ---
dfdx_both = prob5.get_callback('dfdx',
                                input_vars=[{'x': {'units': 'km'}}],
                                output_vars=[{'f': {'units': 'kN'}}])
J_both = dfdx_both(x5)
print(f'df[kN]/dx[km] = {J_both[0, 0]:.4f}  (expected 2.0)')

In [ ]:
assert_near_equal(J_native[0, 0], 2.0)
assert_near_equal(J_kN[0, 0],     2.0 / 1000.0)
assert_near_equal(J_km[0, 0],     2.0 * 1000.0)
assert_near_equal(J_both[0, 0],   2.0)

## Example 7: using `return_index_map` to locate variables in flat vectors

The `create_input_vector()`, `create_output_vector()`, and
`create_jacobian_matrix()` methods all accept a `return_index_map` keyword
argument.  When set to `True`, each method returns a two-element tuple; the
second element is a `dict` that maps variable names to `slice` objects
selecting each variable's elements within the flat array.

This is useful when a callback covers several variables and you want to read
or write a specific variable by name rather than by manually tracking index
offsets.

For `create_jacobian_matrix(return_index_map=True)`, the dict keys are
``(output_name, input_name)`` tuples and the values are
``(row_slice, col_slice)`` tuples that identify the corresponding sub-block
of the Jacobian matrix.

In [ ]:
# --- create_input_vector and create_output_vector with return_index_map ---
# Reuse the 'f' callback from Example 1 (inputs: x, y; output: f_xy).
x_rim, x_map = f.create_input_vector(return_index_map=True)
y_rim, y_map = f.create_output_vector(return_index_map=True)

print('input  index map:', x_map)
print('output index map:', y_map)

# Use the maps to address variables by name without tracking offsets manually.
x_rim[x_map['x']] = 3.0
x_rim[x_map['y']] = -4.0
y_rim = f(x_rim, y=y_rim)
print('f(3, -4) =', y_rim[y_map['f_xy']])

# --- create_jacobian_matrix with return_index_map ---
# Reuse the 'dfdx' callback from Example 2 (inputs: x, y; output: f_xy).
J_rim, J_map = dfdx.create_jacobian_matrix(return_index_map=True)

print('\nJacobian index map:')
for key, slices in J_map.items():
    print(f'  {key}: {slices}')

x2_rim = dfdx.create_input_vector()
x2_rim[0], x2_rim[1] = 1.5, 2.5
dfdx(x2_rim, J=J_rim)

print('df/dx =', J_rim[J_map['f_xy', 'x']])
print('df/dy =', J_rim[J_map['f_xy', 'y']])

In [ ]:
assert x_map == {'x': slice(0, 1), 'y': slice(1, 2)}
assert y_map == {'f_xy': slice(0, 1)}
assert_near_equal(y_rim[y_map['f_xy']], [-15.0])

assert J_map == {('f_xy', 'x'): (slice(0, 1), slice(0, 1)),
                 ('f_xy', 'y'): (slice(0, 1), slice(1, 2))}
assert_near_equal(J_rim[J_map['f_xy', 'x']], [[2*(x2_rim[0]-3.0) + x2_rim[1]]])
assert_near_equal(J_rim[J_map['f_xy', 'y']], [[x2_rim[0] + 2*(x2_rim[1]+4.0)]])

## Example 8: `AnalysisDriver`-like functionality

OpenMDAO's [`AnalysisDriver`](../building_blocks/drivers/analysis_driver) is an OpenMDAO `Driver` that runs an OpenMDAO `Model` for a range of user-specified inputs.
The `Problem` functional interface described here provides a similar capability that might be preferable.
We'll recreate the `Paraboloid` example from the [`AnalysisDriver` docs](../building_blocks/drivers/analysis_driver) here with the functional interface.

The first step is to create the OpenMDAO `Problem`, just like usual:

In [ ]:
# Create and setup the OpenMDAO Problem, just like usual.
from openmdao.test_suite.components.paraboloid import Paraboloid
prob6 = om.Problem()
prob6.model.add_subsystem('comp', Paraboloid(), promotes=['*'])
prob6.setup()

With an `AnalysisDriver` we're usually only interested in the outputs, not derivatives, so we'll use the `f` version of the functional interface:

In [ ]:
# Create the callback function with the inputs and outputs we're interested in.
f = prob6.get_callback('f', input_vars=['x', 'y'], output_vars=['f_xy'])

We can use the fancy [case generating functionality from the `AnalysisDriver` docs](../building_blocks/drivers/analysis_driver.ipynb#generating-cases) with the functional interface.
Here we'll use the `ProductGenerator`:

In [ ]:
# Can still use the fancy `AnalysisGenerator` thingies:
gen = om.ProductGenerator({'x': {'val': [0.0, 0.5, 1.0], 'units': None},
                           'y': {'val': [0.0, 0.5, 1.0], 'units': None}})

Now we can create an input and output vector from the functional interface, and iterate over the cases like this:

In [ ]:
# Create the input and output vectors that we can use for each call.
X, X_idxmap = f.create_input_vector(return_index_map=True)
Y, Y_idxmap = f.create_output_vector(return_index_map=True)

# Iterate over the samples.
for sample in gen:
    # Set inputs.
    X[X_idxmap['x']] = sample['x']['val']
    X[X_idxmap['y']] = sample['y']['val']
    # Call the function, using the pre-allocated output array we created outside the loop.
    f(X, y=Y)
    # Print the results for this sample.
    x = X[X_idxmap['x']][0]
    y = X[X_idxmap['y']][0]
    f_xy = Y[Y_idxmap['f_xy']][0]
    print(f'{x:6.3f}     {y:6.3f}     {f_xy:6.3f}')

## Example 9: Optimization with `scipy.optimize`

The functional interface can be used with external optimizers such as
`scipy.optimize.minimize`.  Using `form='fdfdx'` the callback returns both the
objective value and its gradient in a single model evaluation, which is
the signature expected by `scipy.optimize.minimize` when `jac=True`.

The example below solves the same unconstrained paraboloid minimization as the
[Simple Optimization](../../examples/paraboloid) example — finding the minimum of
$f(x,y) = (x-3)^2 + xy + (y+4)^2 - 3$ — but drives it with
`scipy.optimize.minimize` rather than OpenMDAO's built-in driver.

In [ ]:
import openmdao.api as om
from scipy.optimize import minimize

prob9 = om.Problem()
prob9.model.add_subsystem('paraboloid', om.ExecComp('f = (x-3)**2 + x*y + (y+4)**2 - 3'))
prob9.setup()

# Set initial values.
prob9.set_val('paraboloid.x', 3.0)
prob9.set_val('paraboloid.y', -4.0)
prob9.final_setup()

# Create a callback that returns both f and the gradient in one model evaluation.
fdfdx_opt = prob9.get_callback('fdfdx',
                               input_vars=['paraboloid.x', 'paraboloid.y'],
                               output_vars=['paraboloid.f'])

# scipy.optimize.minimize with jac=True expects the function to return (f, grad).
def objective(x):
    y, J = fdfdx_opt(x)
    return float(y[0]), J[0, :]

x0 = fdfdx_opt.create_input_vector()  # pre-filled with the current problem values
bounds = [(-50, 50), (-50, 50)]

result = minimize(objective, x0, method='SLSQP', jac=True, bounds=bounds)
print('x* =', result.x[0])
print('y* =', result.x[1])
print('f* =', result.fun)

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

assert_near_equal(result.x[0], 6.66666666, tolerance=1.0E-5)
assert_near_equal(result.x[1], -7.33333333, tolerance=1.0E-5)
assert_near_equal(result.fun, -27.33333333, tolerance=1.0E-5)

## Limitations

* **`form='f'` requires explicit variable lists**: unlike the derivative
  forms, `form='f'` cannot fall back to driver variables because no
  `_TotalJacInfo` object is created.  Both `input_vars` and `output_vars`
  must be provided.

* **`run_model()` on every call**: each invocation of the callback calls
  `Problem.run_model()`.  There is no caching; if you need to evaluate
  outputs and derivatives at the same point prefer `form='fdfdx'` over
  separate `'f'` and `'dfdx'` calls.